In [9]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

In [10]:
MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "DataL@b_Cae123"
NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1" 

conf = (
    SparkConf()
    .setAppName("Job_Chargement_Solde")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.extraJavaOptions", "-Dorg.apache.poi.util.IOUtils.setByteArrayMaxOverride=1000000000")
    .set("spark.driver.memory", "16g")
    # AJOUT DES PACKAGES : On ajoute le jar nessie-spark-extensions
    .set("spark.jars.packages", 
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1,""com.crealytics:spark-excel_2.12:3.5.0_0.20.3")
     
     .set("spark.sql.extensions", 
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    # CONFIGURATION DU CATALOGUE NESSIE
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    
    # On définit le bucket bronze comme entrepôt par défaut du catalogue
    .set("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    
    # CONFIGURATION MINIO (S3A)
    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [11]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [ ]:
path_minio = "LECTURE DU FICHIER DEPUIS MINIO"

df_fusionne = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(path_minio)

In [ ]:
df_fusionne.show(5)

+--------------------+--------------------+-----------------+--------------------+--------------------+
|             SERVICE|           ORGANISME|           EMPLOI|               GRADE|      CLASSE/ECHELON|
+--------------------+--------------------+-----------------+--------------------+--------------------+
|               O N I|Min. d'Etat Admin...|             NULL|Non Fonctionnaire...|                NULL|
|Ecole Normale Sup...|Ecole Normale Sup...|             NULL|Non Fonctionnaire...|                NULL|
|Dir.des Aff. Admi...|Min. Affaires Etr...|      AMBASSADEUR|Fonctionnaire Gra...|2ème Classe 3ème ...|
|Ambassade de Brux...|Représentation à ...|      AMBASSADEUR|Fonctionnaire Gra...|2ème Classe 3ème ...|
|      D. G. A. A. T.|Min. d'Etat Admin...|Corps Préfectoral|Fonctionnaire Gra...| AT Grade 1 2ème ech|
+--------------------+--------------------+-----------------+--------------------+--------------------+
only showing top 5 rows



In [ ]:
table_target = "nessie.bronze.extrait_solde"
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")

In [ ]:
df_fusionne.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(table_target)